# Data Processing

1. Imports

In [1]:
# ── 1. Imports ──────────────────────────────────────────────────────────────
import pandas as pd #Processamento e manipulação dos dados
import os # Auxilia o código a manipular pastas para salvar e acessar arquivos

In [2]:
# ── 2. Função de processamento de dados ──────────────────────────────────────────────────────────────

def processamento_dados(arquivo_csv, ano, pasta_destino):
    """
    Processa e limpa os microdados do ENEM a partir de um arquivo CSV, salvando o 
    resultado consolidado em formato Parquet para otimização de leitura e armazenamento.

    O processamento é feito em 'chunks' (lotes de 100.000 linhas) para evitar 
    sobrecarga de memória. As seguintes etapas de limpeza e engenharia de recursos 
    são aplicadas a cada lote:
    
    1. Filtragem: Remoção de treineiros, candidatos ausentes em CN ou MT e notas zero.
    2. Transformação Binária: Mapeamento de gênero ('IN_FEMININO') e acesso à internet.
    3. One-Hot Encoding: Conversão de variáveis categóricas nominais (Raça, Região e 
       Situação de Conclusão) em múltiplas colunas binárias.
    4. Transformação Ordinal: Mapeamento da escolaridade dos pais e renda familiar 
       para escalas numéricas contínuas/ordinais.
    5. Limpeza de Memória: Downcast de tipos numéricos para 'int8' e remoção de 
       colunas intermediárias ou desnecessárias.
    6. Tratamento de Nulos: Exclusão de registros com dados vitais faltantes.

    Args:
        arquivo_csv (str): Caminho (absoluto ou relativo) do arquivo CSV a ser lido.
        ano (int | str): Ano de referência dos dados do ENEM. Será adicionado como 
            uma nova coluna no DataFrame final e usado no nome do arquivo salvo.
        pasta_destino (str): Caminho do diretório onde o arquivo Parquet será salvo. 
            Se o diretório não existir, ele será criado automaticamente.

    Returns:
        pandas.DataFrame: O DataFrame final com todos os lotes processados e concatenados.
            Adicionalmente, salva um arquivo no disco na estrutura: 
            '{pasta_destino}/enem_{ano}_processado.parquet'.
    """
    
    cols_de_interesse = ['TP_FAIXA_ETARIA', 'TP_SEXO', 'TP_COR_RACA', 'TP_ST_CONCLUSAO', 'TP_ESCOLA', 'IN_TREINEIRO', 
                         'TP_LOCALIZACAO_ESC', 'SG_UF_PROVA', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 'NU_NOTA_CN', 
                         'NU_NOTA_MT', 'Q001', 'Q002', 'Q006', 'Q025'] # Seleção das colunas de interesse para processamento
                         
    print(f"Iniciando processamento {ano}...")
    
    if not os.path.exists(pasta_destino): #Cria pasta destino, caso não exista
        os.makedirs(pasta_destino)
        print(f"Pasta {pasta_destino} criada.")

    df = pd.read_csv(arquivo_csv, encoding='latin1', sep=";", on_bad_lines='skip', usecols=cols_de_interesse, chunksize=100000) #Acesso ao arquivo csv
    
    chunks_processados = [] #Cria lista vazia para armazenar os chunks processados
    
    for data in df:  #Loopa os dados para processá-los
        # Filtros
        data = data[data['IN_TREINEIRO'] == 0] #remove os treineiros
        data = data[(data['TP_PRESENCA_CN'] == 1) & (data['TP_PRESENCA_MT'] == 1)]  # Remove as ausências em uma das provas (CN ou MT)
        data = data[(data['NU_NOTA_CN'] != 0) & (data['NU_NOTA_MT'] != 0)] # Elimina dados de participantes que zeraram CN ou MT
        
        # 1. Gênero
        data['IN_FEMININO'] = data['TP_SEXO'].map({'F': 1, 'M': 0}).fillna(-1).astype('int8') # Mapeia gênero, transformando 'F' em 1 e 'M' em 0. Downcast para int8 e fillna para remoção posterior de nulo 

        # 2. Raça
        raca_labels = {0: 'NAO_DECLARADO', 1: 'BRANCA', 2: 'PRETA', 3: 'PARDA', 4: 'AMARELA', 5: 'INDIGENA'} # Mapeia os nomes das raças no lugar de números
        df_raca = pd.get_dummies(data['TP_COR_RACA'].map(raca_labels), prefix='RACA', dtype='int8') # Cria One-Hot Encoding e downcast para int8
        data = pd.concat([data, df_raca], axis=1)
        data.drop('TP_COR_RACA', axis=1, inplace=True) # Remove a coluna original

        # 3. Faixa etária
        data['TP_FAIXA_ETARIA'] = data['TP_FAIXA_ETARIA'].fillna(-1).astype('int8') # Downgrade para int8
        
        # 4. Situação de conclusão
        labels_conclusao = {1: 'CONCLUIDO', 2: 'CONCLUINTE_ANO', 3: 'CONCLUINTE_APOS', 4: 'NAO_CONCLUIDO_NAO_CURSANDO'} # Mapeia os termos dos status de término no lugar de números
        data = data[data['TP_ST_CONCLUSAO'].isin([1, 2])].copy() # copy() para evitar warnings
        df_conclusao = pd.get_dummies(data['TP_ST_CONCLUSAO'].map(labels_conclusao), prefix='ST_CONCLUSAO', dtype='int8') # One-Hot Encoding e downcast para int8
        data = pd.concat([data, df_conclusao], axis=1)
        data.drop('TP_ST_CONCLUSAO', axis=1, inplace=True)  # Remove a coluna original

        # 5. Região 
        map_regiao = {
            'DF': 'CENTRO_OESTE', 'GO': 'CENTRO_OESTE', 'MS': 'CENTRO_OESTE', 'MT': 'CENTRO_OESTE',
            'AL': 'NORDESTE', 'BA': 'NORDESTE', 'CE': 'NORDESTE', 'PB': 'NORDESTE','PE':  'NORDESTE', 'PI': 'NORDESTE', 'RN': 'NORDESTE', 'SE': 'NORDESTE',
            'AC': 'NORTE','AP': 'NORTE', 'AM': 'NORTE', 'MA': 'NORTE', 'PA': 'NORTE', 'RO': 'NORTE', 'RR': 'NORTE', 'TO': 'NORTE',
            'ES': 'SUDESTE', 'MG': 'SUDESTE', 'RJ': 'SUDESTE', 'SP': 'SUDESTE',
            'PR': 'SUL', 'RS': 'SUL', 'SC': 'SUL'
        } 
        data["REGIAO_NOME"] = data["SG_UF_PROVA"].map(map_regiao) # Mapeamento dos estados para sua respectiva região
        df_regiao = pd.get_dummies(data['REGIAO_NOME'], prefix='REGIAO', dtype='int8') # One-Hot Encoding. Downcast para int8 e fillna para remoção posterior de nulo 
        data = pd.concat([data, df_regiao], axis=1)
        data.drop(['SG_UF_PROVA', 'REGIAO_NOME'], axis=1, inplace=True) # Remove a coluna original

        # 6. Escolaridade dos pais (Q001 e Q002)
        map_escolaridade = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 0} # Mapeia as letras da escolaridade para números
        data['ESC_PAI'] = data['Q001'].map(map_escolaridade).fillna(-1).astype('int8') # Downcast para int8 e fillna para remoção posterior de nulo
        data['ESC_MAE'] = data['Q002'].map(map_escolaridade).fillna(-1).astype('int8') # Downcast para int8 e fillna para remoção posterior de nulo
        data = data[(data['ESC_PAI'] >= 0)] # Remoção de nulos
        data = data[(data['ESC_MAE'] >= 0)] # Remoção de nulos

        # 7. Tem internet em casa? (Q025)
        data['TEM_INTERNET'] = data['Q025'].map({'A': 0, 'B': 1}).fillna(-1).astype('int8') # Mapeia se participante tem internet, 0 para 'não' e 1 para 'sim'. Downcast para int8 e fillna para remoção posterior de nulo

        # 8. Renda 
        map_renda = {chr(i): i - 65 for i in range(65, 82)}
        data['RENDA_FAMILIAR'] = data['Q006'].map(map_renda).fillna(-1).astype('int8') # Mapeia as letras que representam as rendas para escala numérica. Downcast para int8 e fillna para remoção posterior de nulo

        # 9. Limpeza Final do Chunk
        # --- FILTRO DE SEGURANÇA (Remove qualquer nulo, representado por -1) ---
        data = data[(data['IN_FEMININO'] != -1) & 
                    (data['TP_FAIXA_ETARIA'] != -1) & 
                    (data['TEM_INTERNET'] != -1) & 
                    (data['RENDA_FAMILIAR'] != -1)]
        cols_para_dropar = ['TP_SEXO', 'TP_ESCOLA', 'IN_TREINEIRO', 'TP_LOCALIZACAO_ESC',
                            'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 'Q001', 'Q002', 
                            'Q006', 'Q025']
        data.drop(columns=cols_para_dropar, inplace=True, errors='ignore')
    
        
        chunks_processados.append(data) # Adiciona os dados na lista de chunks_processados

    
    # Une todos os chunks em um DataFrame final
    df_final = pd.concat(chunks_processados, ignore_index=True)
    df_final['ANO'] = int(ano)
    
    # Salva em parquet na pasta destino
    nome_arquivo = f"enem_{ano}_processado.parquet"
    caminho_final = os.path.join(pasta_destino, nome_arquivo) 
    df_final.to_parquet(caminho_final, index=False, engine='pyarrow', compression='snappy')
    
    print(f"✅ Finalizado {ano}! Arquivo salvo em: {caminho_final}")
    print(f"📊 Total de linhas: {len(df_final)}\n")
    
    return df_final
    


In [3]:
# ── 2. Função de agregação das tabelas ──────────────────────────────────────────────────────────────

def gerar_tabela_master(anos, pasta_origem, pasta_destino):
    """
    Orquestra o processamento em lote dos microdados do ENEM para múltiplos anos 
    e unifica os resultados em um único arquivo Parquet consolidado (Dataset Master).

    A função itera sobre a lista de anos fornecida, busca o CSV correspondente em 
    uma estrutura de diretórios esperada e invoca a função `processamento_dados` 
    para cada arquivo. Ao final do loop, se houver arquivos processados com sucesso, 
    ela lê todos os Parquets individuais, concatena-os e salva um arquivo final master.

    Estrutura esperada do diretório de origem:
        {pasta_origem}/{ano}/dados/MICRODADOS_ENEM_{ano}.csv

    Args:
        anos (list[int] | list[str]): Lista de anos que devem ser processados 
            (ex: [2018, 2019, 2020, 2021, 2022, 2023]).
        pasta_origem (str): Caminho do diretório base onde os arquivos brutos estão.
        pasta_destino (str): Diretório onde os Parquets individuais e o arquivo 
            Master serão salvos. A pasta será criada se não existir.

    Returns:
        pandas.DataFrame | None: Retorna o DataFrame mestre unificado contendo 
            todos os anos processados. Retorna None caso ocorra erro em todos 
            os anos ou os arquivos originais não sejam encontrados.
    """
    if not os.path.exists(pasta_destino):
        os.makedirs(pasta_destino)
    
    arquivos_processados = []  # Lista vazia para controlar quais anos foram processados com sucesso

    for ano in anos:
        caminho_csv = os.path.join(pasta_origem, str(ano), 'dados', f'MICRODADOS_ENEM_{ano}.csv')
        
        if os.path.exists(caminho_csv):
            print(f"\n>>> Processando Ano: {ano}")
            try:
                # Chama a função que já salva em Parquet
                processamento_dados(caminho_csv, ano, pasta_destino)
                
                # Guarda o caminho do arquivo gerado
                caminho_parquet = os.path.join(pasta_destino, f"enem_{ano}_processado.parquet")
                arquivos_processados.append(caminho_parquet)
                
            except Exception as e:
                print(f"❌ Erro ao processar o ano {ano}: {e}") # Levanta erro, caso seja nessário
        else:
            print(f"⚠️ Arquivo não encontrado para o ano {ano}: {caminho_csv}")

    # 2. Agregação Final
    if arquivos_processados:
        print("\n--- Iniciando a unificação de todos os anos ---")
        
        # Lê e concatena todos os arquivos Parquet gerados
        df_master = pd.concat([pd.read_parquet(f) for f in arquivos_processados], ignore_index=True)
        
        # Salva o Dataset Mestre
        caminho_master = os.path.join(pasta_destino, "DATASET_MASTER_ENEM_2018_2023.parquet")
        df_master.to_parquet(caminho_master, index=False, compression='snappy')
        
        print(f"✅ TABELA MASTER CRIADA COM SUCESSO!")
        print(f"📍 Local: {caminho_master}")
        print(f"📊 Total de registros: {len(df_master)}")
        return df_master
    else:
        print("❌ Nenhum arquivo foi processado.")
        return None



In [ ]:
def processamento_dados(arquivo_csv, ano, pasta_destino):

    cols_de_interesse = [
        'TP_FAIXA_ETARIA','TP_SEXO','TP_COR_RACA','TP_ST_CONCLUSAO',
        'IN_TREINEIRO','TP_LOCALIZACAO_ESC','SG_UF_PROVA','TP_PRESENCA_CN',
        'TP_PRESENCA_MT','NU_NOTA_CN','NU_NOTA_MT','Q001','Q002','Q006','Q025'
    ]

    print(f"Iniciando processamento {ano}")

    df = pd.read_csv(
        arquivo_csv,
        encoding='latin1',
        sep=";",
        on_bad_lines='skip',
        usecols=cols_de_interesse,
        chunksize=100000
    )

    chunks_processados = []

    for data in df:

        # filtros
        data = data[data['IN_TREINEIRO'] == 0]
        data = data[(data['TP_PRESENCA_CN'] == 1) & (data['TP_PRESENCA_MT'] == 1)]
        data = data[(data['NU_NOTA_CN'] != 0) & (data['NU_NOTA_MT'] != 0)]

        # genero
        data['IN_FEMININO'] = data['TP_SEXO'].map({'F':1,'M':0}).astype('int8')

        # raça
        raca_labels = {
            1:'BRANCA',2:'PRETA',3:'PARDA',4:'AMARELA',5:'INDIGENA'
        }

        df_raca = pd.get_dummies(
            data['TP_COR_RACA'].map(raca_labels),
            prefix='RACA',
            dtype='int8'
        )

        data = pd.concat([data, df_raca], axis=1)

        # escolaridade pais
        map_escolaridade = {
            'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7
        }

        data['ESC_PAI'] = data['Q001'].map(map_escolaridade).astype('int8')
        data['ESC_MAE'] = data['Q002'].map(map_escolaridade).astype('int8')

        data = data[data['ESC_PAI'] != 7]
        data = data[data['ESC_MAE'] != 7]

        # internet
        data['TEM_INTERNET'] = data['Q025'].map({'A':0,'B':1}).astype('int8')

        cols_para_dropar = [
            'TP_SEXO','TP_COR_RACA','TP_ST_CONCLUSAO',
            'IN_TREINEIRO','SG_UF_PROVA','TP_PRESENCA_CN','TP_PRESENCA_MT',
            'Q001','Q002','Q006','Q025'
        ]

        data.drop(columns=cols_para_dropar, inplace=True, errors='ignore')

        chunks_processados.append(data)

    df_final = pd.concat(chunks_processados, ignore_index=True)

    df_final['ANO'] = ano

    if not os.path.exists(pasta_destino):
        os.makedirs(pasta_destino)

    nome_arquivo = f"enem_{ano}_processado.parquet"

    caminho_final = os.path.join(pasta_destino, nome_arquivo)

    df_final.to_parquet(
        caminho_final,
        index=False,
        engine='pyarrow',
        compression='snappy'
    )

    print(f"Finalizado {ano}")
    print(f"Linhas: {len(df_final)}")

    return caminho_final

In [4]:
# 1. Defina a lista de anos que você quer processar
lista_anos = [2018, 2019, 2020, 2021, 2022, 2023]

# 2. Configure os caminhos das pastas (usando r"..." para evitar erro de barras no Windows)
# Onde estão as pastas 2018, 2019... com os CSVs originais
caminho_dos_dados_crus = r"C:\Users\Claudia\Documents\TCC\data\raw"

# Onde os arquivos Parquet processados e a Tabela Master serão salvos
caminho_dos_dados_limpos = r"C:\Users\Claudia\Documents\TCC\data\processed"

# 3. CHAME APENAS A FUNÇÃO MESTRE
# Ela vai percorrer a lista_anos, chamar processamento_dados para cada um e unir tudo no final.
df_final = gerar_tabela_master(
    anos=lista_anos, 
    pasta_origem=caminho_dos_dados_crus, 
    pasta_destino=caminho_dos_dados_limpos
)

# 4. Verificação rápida
if df_final is not None:
    print("\n--- Conferência de Dados ---")
    print(df_final.head())  # Mostra as primeiras 5 linhas
    print(df_final['ANO'].value_counts().sort_index()) # Mostra quantos candidatos por ano

⚠️ Arquivo não encontrado para o ano 2018: C:\Users\Claudia\Documents\TCC\data\raw\2018\dados\MICRODADOS_ENEM_2018.csv
⚠️ Arquivo não encontrado para o ano 2019: C:\Users\Claudia\Documents\TCC\data\raw\2019\dados\MICRODADOS_ENEM_2019.csv
⚠️ Arquivo não encontrado para o ano 2020: C:\Users\Claudia\Documents\TCC\data\raw\2020\dados\MICRODADOS_ENEM_2020.csv
⚠️ Arquivo não encontrado para o ano 2021: C:\Users\Claudia\Documents\TCC\data\raw\2021\dados\MICRODADOS_ENEM_2021.csv
⚠️ Arquivo não encontrado para o ano 2022: C:\Users\Claudia\Documents\TCC\data\raw\2022\dados\MICRODADOS_ENEM_2022.csv
⚠️ Arquivo não encontrado para o ano 2023: C:\Users\Claudia\Documents\TCC\data\raw\2023\dados\MICRODADOS_ENEM_2023.csv
❌ Nenhum arquivo foi processado.


In [ ]:
lista_anos = [2018, 2019, 2020, 2021, 2022, 2023]
df2020 = r'C:/Users/Claudia/Documents/TCC/data/raw/2019/dados/MICRODADOS_ENEM_2019.csv'
processamento_dados(df2019, 2019, r'data/processed')

In [6]:
# ==========================================
# BLOCO DE EXECUÇÃO PRINCIPAL (PIPELINE)
# ==========================================

# 1. Defina a lista de anos que você deseja processar
anos_enem = [2018, 2019, 2020, 2021, 2022, 2023]

# 2. Defina os caminhos das pastas originais e de destino
# ATENÇÃO: Use o 'r' antes das aspas para o Windows ler as barras \ corretamente
caminho_raw = r"C:\Users\Claudia\Documents\tcc-stem-genero\data\raw"
caminho_processed = r"C:\Users\Claudia\Documents\tcc-stem-genero\data\processed"

# 3. Chame a função orquestradora
print("🚀 Iniciando o Pipeline de Dados do ENEM (2018 a 2023)...")
df_tabela_mestre = gerar_tabela_master(
    anos=anos_enem, 
    pasta_origem=caminho_raw, 
    pasta_destino=caminho_processed
)

# 4. Verificação Rápida após o processamento
if df_tabela_mestre is not None:
    print("\n🎉 Processamento totalmente concluído!")
    print(f"-> Dimensões finais da Tabela: {df_tabela_mestre.shape[0]} linhas e {df_tabela_mestre.shape[1]} colunas.")
    
    print("\n-> Distribuição de candidatos por ano:")
    print(df_tabela_mestre['ANO'].value_counts().sort_index())
    
    print("\n-> Amostra dos dados:")
    display(df_tabela_mestre.head()) # Use print() se não estiver no Jupyter/Colab

🚀 Iniciando o Pipeline de Dados do ENEM (2018 a 2023)...

>>> Processando Ano: 2018
Iniciando processamento 2018...
✅ Finalizado 2018! Arquivo salvo em: C:\Users\Claudia\Documents\tcc-stem-genero\data\processed\enem_2018_processado.parquet
📊 Total de linhas: 3376080


>>> Processando Ano: 2019
Iniciando processamento 2019...
✅ Finalizado 2019! Arquivo salvo em: C:\Users\Claudia\Documents\tcc-stem-genero\data\processed\enem_2019_processado.parquet
📊 Total de linhas: 3168425


>>> Processando Ano: 2020
Iniciando processamento 2020...
✅ Finalizado 2020! Arquivo salvo em: C:\Users\Claudia\Documents\tcc-stem-genero\data\processed\enem_2020_processado.parquet
📊 Total de linhas: 2212904


>>> Processando Ano: 2021
Iniciando processamento 2021...
✅ Finalizado 2021! Arquivo salvo em: C:\Users\Claudia\Documents\tcc-stem-genero\data\processed\enem_2021_processado.parquet
📊 Total de linhas: 1868828


>>> Processando Ano: 2022
Iniciando processamento 2022...
✅ Finalizado 2022! Arquivo salvo em: C:\

,TP_FAIXA_ETARIA,NU_NOTA_CN,NU_NOTA_MT,IN_FEMININO,RACA_AMARELA,RACA_BRANCA,RACA_INDIGENA,RACA_NAO_DECLARADO,RACA_PARDA,RACA_PRETA,...,REGIAO_CENTRO_OESTE,REGIAO_NORDESTE,REGIAO_NORTE,REGIAO_SUDESTE,REGIAO_SUL,ESC_PAI,ESC_MAE,TEM_INTERNET,RENDA_FAMILIAR,ANO
0,2,534.2,496.3,1,1,0,0,0,0,0,...,0,0,1,0,0,4,4,1,5,2018
1,3,506.9,440.6,1,1,0,0,0,0,0,...,0,0,1,0,0,3,4,0,1,2018
2,3,470.6,410.4,0,0,0,0,1,0,0,...,0,0,0,1,0,1,1,0,1,2018
3,2,588.9,711.5,1,0,0,0,0,1,0,...,0,0,1,0,0,5,5,1,8,2018
4,11,477.8,543.1,1,0,0,1,0,0,0,...,0,1,0,0,0,0,4,0,2,2018
